##  ENTREGÁVEL 2: Integração com PostgreSQL
Notebook para carregar os dados no banco

### Conexão com o banco de dados

In [1]:
from spark_utils import get_spark
from db_utils import get_pg_connection, persist_dataframe, list_tables

spark = get_spark("PostgresLoad")
conn = get_pg_connection()
print("Conexão com banco de dados PostgreSQL estabelecida")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/10 20:29:25 WARN Utils: Your hostname, Davis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.200.60.101 instead (on interface en0)
25/11/10 20:29:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/10 20:29:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/10 20:29:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/10 20:29:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/10 20:29:26 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Conexão com banco de dados estabelecida


### Carregamento da camada de dados Silver

In [2]:
df_limpo = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("data/silver/dados_limpos.csv")
)
print(f"Dados Silver carregados: {df_limpo.count()} registros")


Dados Silver carregados: 10476 registros


###  Salvamento da camada Silver no banco de dados

In [3]:
persist_dataframe(df_limpo, "projeto_final", conn)


Tabela projeto_final criada: 10476 registros


###  Carregamento dos dados agregados Gold

In [4]:
metricas = spark.read.option("header", True).option("inferSchema", True).csv("data/gold/metricas_estado.csv")
ativos = spark.read.option("header", True).option("inferSchema", True).csv("data/gold/ativos_patrimonio.csv")

persist_dataframe(metricas, "metricas_estado", conn)
persist_dataframe(ativos, "ativos_patrimonio", conn)


Tabela metricas_estado criada: 5 registros
Tabela ativos_patrimonio criada: 5 registros


### Criação do relacionamento entre as tabelas

In [5]:
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS clientes (
    id_cliente INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome_cliente TEXT,
    email TEXT,
    cidade TEXT,
    estado TEXT
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS ativos (
    id_ativo INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    descricao TEXT,
    valor_atual DOUBLE PRECISION,
    categoria TEXT,
    id_cliente INTEGER,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS metricas_estado (
    estado TEXT PRIMARY KEY,
    total_clientes INTEGER,
    media_patrimonio DOUBLE PRECISION,
    indice_atividade DOUBLE PRECISION
);
""")
conn.commit()
print("Tabelas GOLD estruturadas e relacionadas criadas com sucesso!")


Tabelas GOLD estruturadas e relacionadas criadas com sucesso!


### Verificação das tabelas criadas

In [6]:
for table_name in list_tables(conn):
    print(f" - {table_name}")
conn.close()
print("Tabelas relacionadas criadas e salvas com sucesso!")


 - clientes
 - ativos
 - projeto_final
 - metricas_estado
 - ativos_patrimonio
Tabelas relacionadas criadas e salvas com sucesso!
